**Existen grupos de agentes del Mercado Eléctrico Mayorista con patrones de demanda mensual similares (en nivel, estacionalidad y variabilidad), y en qué medida esos grupos coinciden con categorías reales del sistema eléctrico (tipo de agente, región, categoría tarifaria)?**

----------------------------------

Crear tabla intermedia con **agente + mes + demanda**

↓

Usar agentes con todos los meses con info.

Con columnas nuevas (a partir de **feature** engineering) definir variables como **demanda media, coeficiente de variación, ratio máx/media, tendencia, etc.**

↓

Crear tabla donde los **objetos sean los agentes** y las features las variables previamente definidas.

↓

Con esos datos entrenar al modelo para que sea capaz de identificar **clusters**.

↓

Comparar integrantes de cluster con tipos de agentes reales.

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**1-BASE DE DATOS ORIGINAL**

In [ ]:
##Traemos la base de datos que vamos a usar

ruta = "/content/drive/MyDrive/DATA SCIENCE/TP_DS_10/DF - Hoja 1.csv"
df = pd.read_csv(ruta)

In [ ]:
df.head(15)

,AÑO,MES,AGENTE NEMO,AGENTE DESCRIPCION,TIPO AGENTE,REGION,PROVINCIA,CATEGORIA AREA,CATEGORIA DEMANDA,CATEGORIA TARIFA,TARIFA,TIPO DEMANDA,DEMANDA [MWh]
0,2022,1/1/2022,AARGTAOY,AEROP ARG 2000 - Aeroparque,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"1.537,96"
1,2022,1/1/2022,ABRILHCY,ABRIL CLUB DE CAMPO,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"1.676,53"
2,2022,1/1/2022,ACARQQ3Y,ASOC.COOP.ARG. - Quequén,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"374,89"
3,2022,1/1/2022,ACARSLSY,ASOC.COOP.ARG. - San Lorenzo,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"299,14"
4,2022,1/1/2022,ACERBR1Y,ACERBRAG-Bragado,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"20.723,27"
5,2022,1/1/2022,ACINTBOY,ACINDAR PTA. TABLADA,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"1.624,82"
6,2022,1/1/2022,ACINVCSZ,ACINDAR PTA. V. CONSTITUCION,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"46.166,75"
7,2022,1/1/2022,ACUYLCMY,ACEROS CUYANOS S.A.- L.DE CUYO,GU,CUYO,MENDOZA,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"476,74"
8,2022,1/1/2022,AEEZECCY,AEROP ARG 2000 EX FAA EZEIZA,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"4.034,09"
9,2022,1/1/2022,AGEABACY,ARTE GRAFICO EDIT. ARG. SA,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,Industrial/Comercial Grande,GUMAS/AUTOGENERADORES,GUMA,"934,34"


**2-EDA**

In [ ]:
##Dimensiones de la base de datos original

df.shape

(56696, 13)

In [ ]:
##Nulos por columna

df.isnull().sum()

,0
AÑO,0
MES,0
AGENTE NEMO,0
AGENTE DESCRIPCION,0
TIPO AGENTE,0
REGION,0
PROVINCIA,0
CATEGORIA AREA,0
CATEGORIA DEMANDA,0
CATEGORIA TARIFA,0


In [ ]:
##Cantidad de categrorías de variables por columnas

df.nunique()

,0
AÑO,4
MES,47
AGENTE NEMO,557
AGENTE DESCRIPCION,547
TIPO AGENTE,5
REGION,9
PROVINCIA,22
CATEGORIA AREA,11
CATEGORIA DEMANDA,2
CATEGORIA TARIFA,3


**DICCIONARIO**

**NEMO** = identificador alfanumérico único que se le asigna a cada agente inscripto en el Mercado Eléctrico Mayorista (MEM) para registrar sus transacciones, consumos o inyecciones de energía.

**DESCRIPCION** = nombre completo del agente.



In [ ]:
##TIPO DE AGENTE = clasificación según rol en el mercado.

df['TIPO AGENTE'].unique()

array(['GU', 'AG', 'DI', 'AR', 'DM'], dtype=object)

In [ ]:
##REGION

df['REGION'].unique()

array(['GRAN BS.AS.', 'BUENOS AIRES', 'LITORAL', 'CUYO', 'NORESTE',
       'COMAHUE', 'NOROESTE', 'CENTRO', 'PATAGONICA'], dtype=object)

In [ ]:
##PROVINCIA

df['PROVINCIA'].unique()

array(['BUENOS AIRES', 'SANTA FE', 'MENDOZA', 'MISIONES', 'LA PAMPA',
       'SALTA', 'CORDOBA', 'TUCUMAN', 'CATAMARCA', 'SAN LUIS', 'JUJUY',
       'CHUBUT', 'NEUQUEN', 'RIO NEGRO', 'ENTRE RIOS', 'LA RIOJA',
       'SGO.DEL ESTERO', 'SAN JUAN', 'CORRIENTES', 'CHACO', 'SANTA CRUZ',
       'FORMOSA'], dtype=object)

In [ ]:
##CATEGORIA AREA = distribuidora/zona.

df['CATEGORIA AREA'].unique()

array(['Gran Usuario MEM', 'Resto', 'Entre Rios', 'Mendoza', 'Edeaba',
       'Eden', 'Edenor', 'Edesur', 'Tucuman', 'Cordoba', 'Santa Fe'],
      dtype=object)

In [ ]:
##CATEGORIA DEMANDA = clasif. según demanda mayorista.

df['CATEGORIA DEMANDA'].unique()

array(['Gran Usuario', 'Distribuidor'], dtype=object)

In [ ]:
##CATEGORIA TARIFA = segmento tarifario.

df['CATEGORIA TARIFA'].unique()

array(['Industrial/Comercial Grande', 'Comercial', 'Residencial'],
      dtype=object)

In [ ]:
##TARIFA = tarifa específica aplicada.

df['TARIFA'].unique()

array(['GUMAS/AUTOGENERADORES', 'TARIFA USUARIO NO RESIDENCIAL <300KWH',
       'TARIFA USUARIO NO RESIDENCIAL >=300KWH',
       'TARIFA USUARIO NO RESIDENCIAL >=300KWH S Y E', 'RESIDENCIAL',
       'TARIFA BASE ELECTRODEPEND Y BOMBEROS VOLUNTAR', 'GUMES/GUPAS',
       'RESIDENCIAL NIVEL 1', 'RESIDENCIAL NIVEL 2',
       'RESIDENCIAL NIVEL 3 BASE', 'RESIDENCIAL NIVEL 3 EXCEDENTE',
       'CLUBES DE BARRIO Y PRODUCCION AGRICOLA',
       'TARIFA USU.NO RES. HASTA 10 KW Y > 800KWH/MES',
       'TARIFA USU.NO RES. HASTA 10KW Y <=800KWH/MES',
       'TARIFA USUARIO NO RES.>10 KW Y < DE 300 KW', 'Alumbrado Publico',
       'RESIDENCIAL NIVEL 2 BASE', 'RESIDENCIAL NIVEL 2 EXCEDENTE'],
      dtype=object)

In [ ]:
##TIPO DEMANDA = clasif. según demanda mayorista.

df['TIPO DEMANDA'].unique()

array(['GUMA', 'No Residencial (ESTACIONALIZADA)', 'GUDI',
       'Residencial (ESTACIONALIZADA)', 'GUME'], dtype=object)

**Tipos de datos**

In [ ]:
df.dtypes

,0
AÑO,int64
MES,object
AGENTE NEMO,object
AGENTE DESCRIPCION,object
TIPO AGENTE,object
REGION,object
PROVINCIA,object
CATEGORIA AREA,object
CATEGORIA DEMANDA,object
CATEGORIA TARIFA,object


In [ ]:
df['DEMANDA [MWh]'].head(10)

,DEMANDA [MWh]
0,"1.537,96"
1,"1.676,53"
2,"374,89"
3,"299,14"
4,"20.723,27"
5,"1.624,82"
6,"46.166,75"
7,"476,74"
8,"4.034,09"
9,"934,34"


In [ ]:
##Transformación de Demanda a float, inicialmnete tiró error porque había valores "-" esos se transforman en 0, ya que se cosidera que es mes no hubo demanda (si no existiera el dato, la fila no debería estar, ya que s una base de datos de demanda mensual)

if df['DEMANDA [MWh]'].dtype == 'object':
    df['DEMANDA [MWh]'] = (
        df['DEMANDA [MWh]']
        .astype(str)
        .str.strip()
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    df['DEMANDA [MWh]'] = pd.to_numeric(df['DEMANDA [MWh]'], errors='coerce').fillna(0)
    df['DEMANDA [MWh]'] = df['DEMANDA [MWh]'].astype('float64')

print(df['DEMANDA [MWh]'].dtype)
print("Nulos: ", df['DEMANDA [MWh]'].isnull().sum())
df['DEMANDA [MWh]'].head(10)


float64
Nulos:  0


,DEMANDA [MWh]
0,1537.96
1,1676.53
2,374.89
3,299.14
4,20723.27
5,1624.82
6,46166.75
7,476.74
8,4034.09
9,934.34


In [ ]:
Porcentaje_Nulos = 100*(df['DEMANDA [MWh]'].isnull().sum())/len(df['DEMANDA [MWh]'])
print(Porcentaje_Nulos,"%")

## estos nulos se van a eliminar de la tabla intermedia

0.0 %


In [ ]:
##Transformación de MES a formato fecha

df['MES'] = pd.to_datetime(df['MES'], format='%d/%m/%Y')
df['MES'].unique()

<DatetimeArray>
['2022-01-01 00:00:00', '2022-02-01 00:00:00', '2022-03-01 00:00:00',
 '2022-04-01 00:00:00', '2022-05-01 00:00:00', '2022-06-01 00:00:00',
 '2022-07-01 00:00:00', '2022-08-01 00:00:00', '2022-09-01 00:00:00',
 '2022-10-01 00:00:00', '2022-11-01 00:00:00', '2022-12-01 00:00:00',
 '2023-01-01 00:00:00', '2023-02-01 00:00:00', '2023-03-01 00:00:00',
 '2023-04-01 00:00:00', '2023-05-01 00:00:00', '2023-06-01 00:00:00',
 '2023-07-01 00:00:00', '2023-08-01 00:00:00', '2023-09-01 00:00:00',
 '2023-10-01 00:00:00', '2023-11-01 00:00:00', '2023-12-01 00:00:00',
 '2024-01-01 00:00:00', '2024-02-01 00:00:00', '2024-03-01 00:00:00',
 '2024-04-01 00:00:00', '2024-05-01 00:00:00', '2024-06-01 00:00:00',
 '2024-07-01 00:00:00', '2024-08-01 00:00:00', '2024-09-01 00:00:00',
 '2024-10-01 00:00:00', '2024-11-01 00:00:00', '2024-12-01 00:00:00',
 '2025-01-01 00:00:00', '2025-02-01 00:00:00', '2025-03-01 00:00:00',
 '2025-04-01 00:00:00', '2025-05-01 00:00:00', '2025-06-01 00:00:00',
 '20

**TABLA INTERMEDIA DE AGENTE+MES+DEMANDA [AMD]**

In [ ]:
AMD = df.groupby(['AGENTE NEMO', 'MES'], as_index= False)['DEMANDA [MWh]'].sum()
display(AMD)

,AGENTE NEMO,MES,DEMANDA [MWh]
0,AARGTAOY,2022-01-01,1537.96
1,AARGTAOY,2022-02-01,1201.69
2,AARGTAOY,2022-03-01,1246.53
3,AARGTAOY,2022-04-01,1098.70
4,AARGTAOY,2022-05-01,1186.89
...,...,...,...
22802,YPFTREUA,2022-06-01,0.00
22803,YPFTREUA,2022-07-01,0.00
22804,YPFTREUA,2022-08-01,0.00
22805,YPFTREUA,2022-09-01,0.00


In [ ]:
AMD.shape

(22807, 3)

In [ ]:
meses_por_agente = AMD.groupby('AGENTE NEMO')['MES'].nunique()
meses_por_agente

,MES
AGENTE NEMO,
AARGTAOY,47
ABRILHCY,47
ACARQQ3Y,47
ACARSLSY,47
ACERBR1Y,47
...,...
YPFPHUQA,47
YPFPTRZZ,47
YPFSECZY,47


In [ ]:
##Evalución de cuántos agentes sin los 47 meses de análisis.

agentes_incompletos = meses_por_agente[meses_por_agente < 47]
agentes_completos = meses_por_agente[meses_por_agente == 47]
print(100* len(agentes_incompletos)/len(agentes_incompletos+agentes_completos))


23.33931777378815


In [ ]:
##Se deberían eliminar el 23% de los agentes.
##Se pueden considerar agentes con al menos 12 meses (es decir, 1 año completo, lo que permitiría calcular estacionalidad)

agentes_incompletos1 = meses_por_agente[meses_por_agente < 12]
agentes_completos1 = meses_por_agente[meses_por_agente >= 12]
print(100* len(agentes_incompletos1)/len(agentes_incompletos1+agentes_completos1))



7.719928186714542


In [ ]:
agentes_incluidos = meses_por_agente[meses_por_agente >= 12]

AMD_incluidos = AMD[AMD['AGENTE NEMO'].isin(agentes_incluidos.index)]

AMD_incluidos

,AGENTE NEMO,MES,DEMANDA [MWh]
0,AARGTAOY,2022-01-01,1537.96
1,AARGTAOY,2022-02-01,1201.69
2,AARGTAOY,2022-03-01,1246.53
3,AARGTAOY,2022-04-01,1098.70
4,AARGTAOY,2022-05-01,1186.89
...,...,...,...
22792,YPFTORUZ,2025-07-01,0.00
22793,YPFTORUZ,2025-08-01,0.00
22794,YPFTORUZ,2025-09-01,0.00
22795,YPFTORUZ,2025-10-01,0.00
